In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members import ExpectColumnValuesToHaveListMembers
from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange
from expectations.expect_column_values_to_have_unique_list_members import ExpectColumnValuesToHaveUniqueListMembers

# Create Expectation Suite for Disease Correlation Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# preprod
disease_correlation_file = syn.get("syn73774521").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(disease_correlation_file)

module_columns = ["IFG", "PHG", "TCX", "CBE", "DLPFC", "FP", "STG"]

# A module key is omitted entirely when that brain region is absent for a row, so pd.read_json
# loads those cells as NaN. Coerce non-dict cells to None so they serialize to JSON null (not the
# invalid string "NaN") and satisfy the absent-module schema after convert_nested_columns_to_json.
for col in module_columns:
    df[col] = df[col].apply(lambda v: v if isinstance(v, dict) else None)

df = GreatExpectationsRunner.convert_nested_columns_to_json(df, module_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "disease_correlation"

## Add Expectations to Validator Object For Each Column

In [ ]:
# Fails on a new/unknown column, or a missing expected one.
validator.expect_table_columns_to_match_set(
    column_set=[
        "name", "matched_control", "model_type", "modified_genes", "cluster", "age", "age_numeric", "sex",
        "IFG", "PHG", "TCX", "CBE", "DLPFC", "FP", "STG",
    ],
    exact_match=True,
)

In [ ]:
# name (be_in_set values aligned with ui_config dc_model_name_filter_vals)
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")
validator.expect_column_values_to_be_in_set(
    "name",
    [
        "5xFAD (IU/Jax/Pitt)", "APOE4", "LOAD1", "LOAD1.Abca7A1527G", "LOAD1.Ceacam1KO",
        "LOAD1.Clasp2L163P", "LOAD1.CR1long", "LOAD1.Il1rapExon2KO", "LOAD1.MthfrC677T",
        "LOAD1.Snx1D465N", "Trem2R47H",
    ],
)

In [ ]:
# matched_control
validator.expect_column_values_to_be_of_type("matched_control", "str")
validator.expect_column_values_to_not_be_null("matched_control")
validator.expect_column_values_to_be_in_set("matched_control", ["C57BL/6J"])

In [ ]:
# model_type (be_in_set values aligned with ui_config model_type_filter_vals)
validator.expect_column_values_to_be_of_type("model_type", "str")
validator.expect_column_values_to_not_be_null("model_type")
validator.expect_column_values_to_be_in_set("model_type", ["Familial AD", "Late Onset AD"])

In [ ]:
# cluster (be_in_set values aligned with ui_config dc_menu2)
validator.expect_column_values_to_be_of_type("cluster", "str")
validator.expect_column_values_to_not_be_null("cluster")
validator.expect_column_values_to_be_in_set(
    "cluster",
    [
        "Immune System - Consensus Cluster B",
        "Neuronal System - Consensus Cluster C",
        "Organelle Biogenesis, Cellular Stress Response - Consensus Cluster E",
        "Cell Cycle, NMD Consensus Cluster D",
        "ECM Organization - Consensus Cluster A",
    ],
)

In [ ]:
# age / age_numeric (be_in_set values aligned with ui_config dc_age_filter_vals)
validator.expect_column_values_to_be_of_type("age", "str")
validator.expect_column_values_to_not_be_null("age")
validator.expect_column_values_to_be_in_set("age", ["4 months", "8 months", "12 months"])

validator.expect_column_values_to_be_of_type("age_numeric", "int")
validator.expect_column_values_to_not_be_null("age_numeric")
validator.expect_column_values_to_be_in_set("age_numeric", [4, 8, 12])

In [ ]:
# sex (be_in_set values aligned with ui_config sex_filter_vals)
validator.expect_column_values_to_be_of_type("sex", "str")
validator.expect_column_values_to_not_be_null("sex")
validator.expect_column_values_to_be_in_set("sex", ["Female", "Male"])

In [ ]:
# modified_genes (list_members values aligned with ui_config dc_modified_gene_filter_vals)
validator.expect_column_values_to_be_of_type("modified_genes", "list")
validator.expect_column_values_to_not_be_null("modified_genes")
validator.expect_column_values_to_have_list_length_in_range(column="modified_genes", list_length_range=[1, 4])
validator.expect_column_values_to_have_list_members_of_type(column="modified_genes", member_type="str")
validator.expect_column_values_to_have_unique_list_members(column="modified_genes")
validator.expect_column_values_to_have_list_members(
    column="modified_genes",
    list_members=[
        "Abca7", "APOE", "APP", "Ceacam1", "Clasp2", "CR1", "CR2", "Il1rap", "Mthfr", "PSEN1",
        "Snx1", "Trem2",
    ],
)

## Module Columns

Each brain-region module is either a `{correlation, adj_p_val}` object (both required, non-null
numbers — `correlation` in `[-1, 1]`, `adj_p_val` in `[0, 1]`) or JSON `null` when the module is
absent for that row. A single `oneOf` schema (`module_schema.json`) validates both states per column
and rejects null/missing sub-fields, extra properties, and out-of-range values.

In [ ]:
with open("../src/agoradatatools/great_expectations/gx/json_schemas/disease_correlation/module_schema.json", "r") as file:
    module_schema = json.load(file)

# Each brain-region module is either a {correlation, adj_p_val} object (both required, non-null
# numbers) or JSON null when the module is absent. A single oneOf schema validates both states per
# column and rejects null/missing sub-fields. (Replaces the prior per-cluster present/absent scheme,
# which hardcoded brittle cluster-name strings; this trades per-cluster precision for robustness.)
for col in ["IFG", "PHG", "TCX", "CBE", "DLPFC", "FP", "STG"]:
    validator.expect_column_values_to_match_json_schema(col, json_schema=module_schema)

## Cross-column Uniqueness

In [ ]:
validator.expect_compound_columns_to_be_unique(["name", "cluster", "age", "sex"])

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()